# 02 — TF-IDF Baseline (LIAR)

**Goal:** run a simple, reproducible in-domain baseline on LIAR and report:
- Accuracy
- Macro-F1
- Confusion matrix
- Classification report

**Expected data path (edit if needed):**
- `data/liar/train.tsv`
- `data/liar/valid.tsv`
- `data/liar/test.tsv`

After running, copy outputs into `results/liar_baseline.md`.


## 0) Setup (edit these parameters)

- `LABEL_MODE`: `'six'` (original LIAR labels) or `'binary'` (REAL/FAKE mapping)
- `EXCLUDE_HALF_TRUE`: if `True`, drop `half-true` for a robustness check


In [ ]:
# If running on a fresh environment (optional):
# !pip -q install scikit-learn pandas

import os
from pathlib import Path
from datetime import datetime

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# -----------------
# Parameters
# -----------------
DATA_DIR = Path('data/liar')  # change if your dataset path is different

LABEL_MODE = 'six'          # 'six' or 'binary'
EXCLUDE_HALF_TRUE = False   # only applies when LABEL_MODE == 'binary'

# TF-IDF params
NGRAM_RANGE = (1, 2)
MIN_DF = 2
MAX_DF = 0.9

# Classifier choice: 'lr' or 'svm'
CLF_TYPE = 'lr'
LR_MAX_ITER = 2000

# For reproducibility (only affects some models)
RANDOM_SEED = 42

# Binary mapping (for cross-dataset alignment later)
BIN_MAP = {
    'true': 1,
    'mostly-true': 1,
    'half-true': 1,
    'barely-true': 0,
    'false': 0,
    'pants-fire': 0
}

print('Params:')
print('  DATA_DIR =', DATA_DIR)
print('  LABEL_MODE =', LABEL_MODE)
print('  EXCLUDE_HALF_TRUE =', EXCLUDE_HALF_TRUE)
print('  TF-IDF ngram/min_df/max_df =', NGRAM_RANGE, MIN_DF, MAX_DF)
print('  CLF_TYPE =', CLF_TYPE)


## 1) Load LIAR

This loads `train.tsv`, `valid.tsv`, `test.tsv` and prints shapes + label counts.

In [ ]:
train_path = DATA_DIR / 'train.tsv'
valid_path = DATA_DIR / 'valid.tsv'
test_path  = DATA_DIR / 'test.tsv'

cols = [
    'id','label','statement','subject','speaker','speaker_job','state','party',
    'barely_true_counts','false_counts','half_true_counts','mostly_true_counts',
    'pants_on_fire_counts','context'
]

def load_tsv(p: Path) -> pd.DataFrame:
    if not p.exists():
        raise FileNotFoundError(
            f'File not found: {p}.\n'
            f'Please upload LIAR TSVs to {DATA_DIR}/ or change DATA_DIR.'
        )
    return pd.read_csv(p, sep='\t', header=None, names=cols)

train = load_tsv(train_path)
valid = load_tsv(valid_path)
test  = load_tsv(test_path)

print('Shapes:')
print('  train:', train.shape)
print('  valid:', valid.shape)
print('  test :', test.shape)

print('\nTrain label counts:')
print(train['label'].value_counts())

print('\nSample statements:')
for i, s in enumerate(train['statement'].head(3).astype(str).tolist(), 1):
    print(f'{i}. {s}')


## 2) Prepare labels + text

- Always uses `statement` as text.
- For `LABEL_MODE='six'`, keep the original LIAR labels.
- For `LABEL_MODE='binary'`, map to REAL/FAKE (and optionally exclude `half-true`).

In [ ]:
def clean_text(s: pd.Series) -> pd.Series:
    return s.astype(str).fillna('').str.strip()

def prepare(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['statement'] = clean_text(out['statement'])
    out = out[out['statement'] != '']

    if LABEL_MODE == 'binary':
        out['y'] = out['label'].map(BIN_MAP)
        if EXCLUDE_HALF_TRUE:
            out = out[out['label'] != 'half-true']
        out = out.dropna(subset=['y'])
        out['y'] = out['y'].astype(int)
    else:
        out['y'] = out['label'].astype(str)
    return out

train_p = prepare(train)
test_p  = prepare(test)

print('Prepared shapes:')
print('  train_p:', train_p.shape)
print('  test_p :', test_p.shape)

print('\nLabel distribution (train):')
print(train_p['y'].value_counts())

print('\nLabel distribution (test):')
print(test_p['y'].value_counts())


## 3) Train baseline

Baseline pipeline:
- TF-IDF (1–2 grams)
- Logistic Regression (or Linear SVM)


In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF
)

if CLF_TYPE == 'svm':
    clf = LinearSVC()
else:
    clf = LogisticRegression(max_iter=LR_MAX_ITER)

pipe = Pipeline([
    ('tfidf', vectorizer),
    ('clf', clf)
])

pipe.fit(train_p['statement'], train_p['y'])
pred = pipe.predict(test_p['statement'])

print('Model trained.')


## 4) Evaluate (copy these into `results/liar_baseline.md`)

Reports:
- Accuracy
- Macro-F1
- Confusion matrix
- Classification report


In [ ]:
acc = accuracy_score(test_p['y'], pred)
macro_f1 = f1_score(test_p['y'], pred, average='macro')

print('Accuracy:', acc)
print('Macro-F1:', macro_f1)

print('\nClassification report:')
print(classification_report(test_p['y'], pred, digits=3))

# Confusion matrix
labels_order = sorted(test_p['y'].unique().tolist())
cm = confusion_matrix(test_p['y'], pred, labels=labels_order)

print('\nConfusion matrix (label order):', labels_order)
print(cm)


## 5) Optional: create a small text block you can paste into `results/liar_baseline.md`

This writes a helper file under `results/` in your runtime. You can upload it back to GitHub.

In [ ]:
run_date = datetime.now().strftime('%Y-%m-%d %H:%M')

summary_lines = []
summary_lines.append('# LIAR Baseline Run Output')
summary_lines.append('')
summary_lines.append(f'- Date: {run_date}')
summary_lines.append(f'- LABEL_MODE: {LABEL_MODE}')
summary_lines.append(f'- EXCLUDE_HALF_TRUE: {EXCLUDE_HALF_TRUE}')
summary_lines.append(f'- TF-IDF: ngram_range={NGRAM_RANGE}, min_df={MIN_DF}, max_df={MAX_DF}')
summary_lines.append(f'- Classifier: {"LinearSVC" if CLF_TYPE=="svm" else "LogisticRegression"}')
summary_lines.append('')
summary_lines.append('## Metrics (test split)')
summary_lines.append(f'- Accuracy: {acc}')
summary_lines.append(f'- Macro-F1: {macro_f1}')
summary_lines.append('')
summary_lines.append('## Confusion matrix')
summary_lines.append(f'- Label order: {labels_order}')
summary_lines.append('```')
summary_lines.append(str(cm))
summary_lines.append('```')
summary_lines.append('')
summary_lines.append('## Classification report')
summary_lines.append('```')
summary_lines.append(classification_report(test_p['y'], pred, digits=3))
summary_lines.append('```')

out_dir = Path('results')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'liar_baseline_run_output.md'
out_path.write_text('\n'.join(summary_lines), encoding='utf-8')

print('Saved:', out_path.resolve())
